In [1]:
import re
import os
from itertools import product

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.asyncio import tqdm
import xlwings as xw

In [2]:
path_preprocessado = Path(r"data\input\anexo_c\preprocessado_PIS_ANUAL.csv")
TAX_FILTERS_PIS = {"PIS": ["Total das Receitas", " Exclusão", " Dedução", "Sem efeito"]}
template_PIS_path = Path(r"docs\template_pis_cofins.xlsx")
path_output_PIS_ANUAL = Path(r"data\output\anexo_c\PIS")
dashboard = pd.read_csv(path_preprocessado)

In [3]:
contratos = sorted(dashboard["NumContrato"].unique().tolist())
for contrato in tqdm(contratos, unit=" contrato"):
    tqdm.desc = f"Processando contrato: {contrato}"

    # pega só o que é desse contrato no final.csv
    df = dashboard[dashboard["NumContrato"] == contrato]

    # se não existir no CSV, pula
    if df.empty:
        continue

    num_str = str(contrato).zfill(7)  # completa com zeros à esquerda

100%|██████████| 15/15 [00:00<00:00, 2120.98 contrato/s]


In [4]:
df

,Ano,Descrição,Conta_Nome,Cosif_Nome,ValorCredito,Movimentacao,NumContrato
5616,2001,(01) Total das Receitas,71300010010036 - RENDAS APROPRIADAS-CL2,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,0.00,0.0000,980495
5617,2001,(01) Total das Receitas,71300010010046 - RDAS APRPR.-RI-PORT.140,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,1160.50,1160.5000,980495
5618,2001,(01) Total das Receitas,71300010030016 - SUPERVENIENCIA DE DEPRECIACAO,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,3946.31,3946.3100,980495
5619,2001,(01) Total das Receitas,71300010050016 - COM.PERMANENCIA-CN-R.INT.,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,227.78,30.2800,980495
5620,2001,(01) Total das Receitas,71300010050026 - COM.PERMANENCIA-CL2/CL3-RI,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,0.00,0.0000,980495
...,...,...,...,...,...,...,...
6104,2013,Base de Cálculo (01)-(02)-(03),NaN,Cálculo da Cofins - Alíquota 4%,NaN,0.0000,980495
6105,2014,Base de Cálculo (01)-(02)-(03),NaN,Cálculo da Cofins - Alíquota 4%,NaN,0.0000,980495
6106,2015,Base de Cálculo (01)-(02)-(03),NaN,Cálculo da Cofins - Alíquota 4%,NaN,0.0000,980495
6107,2016,Base de Cálculo (01)-(02)-(03),NaN,Cálculo da Cofins - Alíquota 4%,NaN,0.0000,980495


In [11]:
df_filtrado = df[df["Descrição"]=="(01) Total das Receitas"].pivot_table(
    index=["Conta_Nome"],
    columns="Ano",
    values="Movimentacao",
    aggfunc="sum",
    fill_value=0,
).reset_index()

In [16]:
df

,Ano,Descrição,Conta_Nome,Cosif_Nome,ValorCredito,Movimentacao,NumContrato
5616,2001,(01) Total das Receitas,71300010010036 - RENDAS APROPRIADAS-CL2,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,0.00,0.0000,980495
5617,2001,(01) Total das Receitas,71300010010046 - RDAS APRPR.-RI-PORT.140,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,1160.50,1160.5000,980495
5618,2001,(01) Total das Receitas,71300010030016 - SUPERVENIENCIA DE DEPRECIACAO,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,3946.31,3946.3100,980495
5619,2001,(01) Total das Receitas,71300010050016 - COM.PERMANENCIA-CN-R.INT.,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,227.78,30.2800,980495
5620,2001,(01) Total das Receitas,71300010050026 - COM.PERMANENCIA-CL2/CL3-RI,71210001 - RENDAS DE ARRENDAM.FINANC.- RECURSO...,0.00,0.0000,980495
...,...,...,...,...,...,...,...
6104,2013,Base de Cálculo (01)-(02)-(03),NaN,Cálculo da Cofins - Alíquota 4%,NaN,0.0000,980495
6105,2014,Base de Cálculo (01)-(02)-(03),NaN,Cálculo da Cofins - Alíquota 4%,NaN,0.0000,980495
6106,2015,Base de Cálculo (01)-(02)-(03),NaN,Cálculo da Cofins - Alíquota 4%,NaN,0.0000,980495
6107,2016,Base de Cálculo (01)-(02)-(03),NaN,Cálculo da Cofins - Alíquota 4%,NaN,0.0000,980495


In [14]:
import pandas as pd

def gerar_excel_sem_pivot(pivot, output_path):
    with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
        workbook = writer.book
        sheet = workbook.add_worksheet("PIS_ANUAL")
        writer.sheets["PIS_ANUAL"] = sheet

        # ===== estilos =====
        header_format = workbook.add_format({
            "bold": True,
            "bg_color": "#D9E1F2",
            "border": 1
        })

        number_format = workbook.add_format({
            "num_format": "#,##0.00"
        })

        total_format = workbook.add_format({
            "bold": True,
            "top": 1,
            "num_format": "#,##0.00"
        })

        # ===== pivot no pandas =====
        # pivot = df.pivot_table(
        #     index=["Ano"],
        #     columns="Tributo",
        #     values="Movimentacao",
        #     aggfunc="sum",
        #     fill_value=0,
        # ).reset_index()

        # ===== escreve header =====
        for col_idx, col in enumerate(pivot.columns):
            sheet.write(0, col_idx, col, header_format)

        # ===== escreve dados =====
        for row_idx, row in pivot.iterrows():
            for col_idx, value in enumerate(row):
                sheet.write(row_idx + 1, col_idx, value, number_format)

        # ===== total =====
        last_row = len(pivot) + 1
        sheet.write(last_row, 0, "Grand Total", header_format)

        for col in range(1, len(pivot.columns)):
            col_letter = chr(65 + col)
            formula = f"=SUM({col_letter}2:{col_letter}{last_row})"
            sheet.write_formula(last_row, col, formula, total_format)

        # largura das colunas
        sheet.set_column(0, len(pivot.columns), 18)

In [15]:
gerar_excel_sem_pivot(df_filtrado, path_output_PIS_ANUAL / f"PIS_ANUAL_{num_str}.xlsx")

In [1]:
import math

def calcular_nos(
    total_contratos: int,
    contratos_por_task: int,
    tempo_medio_task_min: float,
    horas_desejadas: float,
    tasks_por_no: int = 1,
) -> dict:
    total_tasks = math.ceil(total_contratos / contratos_por_task)
    tempo_desejado_min = horas_desejadas * 60

    nos = math.ceil(
        (total_tasks * tempo_medio_task_min) /
        (tempo_desejado_min * tasks_por_no)
    )

    return {
        "total_tasks": total_tasks,
        "nos_necessarios": nos
    }

# exemplo
resultado = calcular_nos(
    total_contratos=35000,
    contratos_por_task=100,
    tempo_medio_task_min=15,
    horas_desejadas=2,
    tasks_por_no=2
)

print(resultado)

{'total_tasks': 350, 'nos_necessarios': 22}
